# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 28 · Round 13: Observed motion dynamics

A 12-channel representation, not a new-data or algorithm search. This round is independent of the other supplied round. Do not modify it after inspecting other-round scores. No training occurs in this notebook. Keep observed-feature plots private until review.

In [ ]:
from pathlib import Path
import json, sys, subprocess, signal
import plotly.io as pio
KIT=Path('/home/sagemaker-user/nfl_feature_rounds12_13')
OUT=Path('/home/sagemaker-user/nfl-feature-round13-results')
PY=Path('/home/sagemaker-user/nfl-player-trajectory/.venv/bin/python')
ROUND=13
if not KIT.is_dir() or not PY.is_file():
    raise FileNotFoundError('Open the existing NFL space and extract the combined package first.')
sys.path.insert(0,str(KIT))
import visuals
pio.renderers.default='plotly_mimetype'
def run(stage,*options):
    p=subprocess.Popen([str(PY),str(KIT/'run_round.py'),stage,'--round',str(ROUND),*options],
        stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in p.stdout: print(line,end='')
        code=p.wait()
    except KeyboardInterrupt:
        p.send_signal(signal.SIGINT)
        try:p.wait(timeout=10)
        except subprocess.TimeoutExpired:p.kill();p.wait()
        raise
    if code:
        raise RuntimeError(f'{stage} stopped ({code}). Preserve outputs, export the report, and do not change settings.')
def show(fig,name):
    visuals.save(fig,OUT,name).show()

## Evidence before computation
These three figures use the uploaded Round 11 aggregates. Training fit is not an evaluation gain; unused observed plays do not establish usable extra labels.

In [ ]:
show(visuals.history_fit(KIT),'round11_fit')
show(visuals.coverage(KIT),'round11_coverage')
show(visuals.core_support(KIT,ROUND),'round11_core_support')

## Verify the existing artifact chain
Require `family_preflight_passed`. No synchronization, raw download, package installation, or previous model refit.

In [ ]:
run('preflight')

## Smoke: 32 training plays only
Require `family_smoke_passed`. The first six feature channels must exactly match the Round 11 audited arrays. Targets are not unpacked.

In [ ]:
run('smoke')

## Prepare only after smoke succeeds
Require `family_features_ready`. Existing checkpoints are reused. Training-only support and range statistics; evaluation feature values are constructed without opening their targets. Do not clip or rescale after reviewing plots.

In [ ]:
run('prepare')
show(visuals.support(OUT,ROUND),'feature_support')
show(visuals.ranges(OUT,ROUND),'feature_ranges')

## Stop here before scientific fitting
Save this notebook, then open notebook 29. Both rounds retain the same 699 training plays and 152 evaluation plays. Data expansion is deliberately not mixed into feature attribution. No predictive benefit has been measured by this notebook.